# EDA — MusicCaps & FMA-small

Quick exploratory checks used before building the pipeline:
- MusicCaps caption length / aspect_list frequency distribution (top-50 tag vocabulary selection)
- FMA-small genre balance and official split sizes (6400/800/800)
- Sanity-check audio loading (librosa) and chroma extraction on a handful of clips

See `config.yaml` for dataset paths and `src/audio_features.py` for the feature-extraction
functions used here.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve() / 'src'))
import pandas as pd
import yaml

cfg = yaml.safe_load(open('../config.yaml'))
cfg['data']

In [ ]:
# Load MusicCaps and inspect aspect_list frequency (top-50 vocabulary selection)
import ast
from collections import Counter

df = pd.read_csv(cfg['data']['musiccaps_csv'])
print(df.shape)
df.head()

In [ ]:
aspect_counter = Counter()
for row in df['aspect_list']:
    try:
        aspects = ast.literal_eval(row)
    except Exception:
        continue
    aspect_counter.update(a.strip() for a in aspects)

top50 = aspect_counter.most_common(cfg['data']['num_tag_vocab'])
top50

In [ ]:
# FMA-small genre balance / official split sizes
tracks = pd.read_csv(cfg['data']['fma_tracks_csv'], index_col=0, header=[0, 1])
small = tracks[tracks[('set', 'subset')] == 'small']
print(small.shape)
small[('set', 'split')].value_counts()

In [ ]:
# Audio loading + chroma sanity check on one clip
from audio_features import load_audio, chroma_sequence

# y = load_audio('path/to/a/clip.mp3')
# chroma_sequence(y).shape  # (num_windows, 12)